In [ ]:
# Import core data-handling and numerical libraries
import pandas
import torch
import glob, os

# Import dataset utilities and training wrapper from HugsVision
#   - VisionDataset handles image-folder loading, balancing, and augmentation
#   - VisionClassifierTrainer wraps HuggingFace's Trainer with defaults for vision tasks
from hugsvision.dataio.VisionDataset import VisionDataset
from hugsvision.nnet.VisionClassifierTrainer import VisionClassifierTrainer

# Import model and preprocessing components from HuggingFace Transformers
#   - ViTForImageClassification: Vision Transformer head adapted for classification
#   - AutoImageProcessor: unified image preprocessor (preferred over deprecated FeatureExtractor)
#   - ViTFeatureExtractor remains imported for backward compatibility with older utilities
from transformers import ViTFeatureExtractor, ViTForImageClassification, AutoImageProcessor

# Import lightweight inference wrapper for convenient single-image predictions
from hugsvision.inference.VisionClassifierInference import VisionClassifierInference

# Import pandas for tabular logging and result aggregation
import pandas as pd

# Import seaborn and matplotlib for exploratory data analysis and diagnostics/visualization
import seaborn as sns
import matplotlib.pyplot as plt

# Import confusion matrix utility for evaluating classification performance across classes
from sklearn.metrics import confusion_matrix


In [ ]:
# -- Hardware sanity check ------------------------------------------------------
# Purpose: verify that PyTorch detects a CUDA-capable GPU before training.
# Rationale: GPU availability dramatically affects training speed; explicitly
#            checking avoids silently falling back to the CPU.
# Note: if `torch.cuda.is_available()` returns False, training will proceed on CPU.
print("CUDA disponible:", torch.cuda.is_available())

# If a CUDA device is present, also report the specific GPU model for reproducibility
# and to facilitate debugging/performance attribution in experiment logs.
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA disponible: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [ ]:
# -- Data preparation -----------------------------------------------------------
# Load an image classification dataset from a folder structure:
#   root/
#     class_0/
#       img_0.jpg, img_1.jpg, ...
#     class_1/
#       ...
# The loader performs a stratified split, optional class balancing, and optional
# data augmentation suitable for training Vision Transformers.
train, val, id2label, label2id = VisionDataset.fromImageFolder(
    "./train/",      # Root directory containing one subfolder per class
    test_ratio=0.1,  # Hold out 10% of the data for validation
    balanced=True,   # Rebalance classes to mitigate label imbalance in training
    augmentation=True,    # Apply standard image augmentations during training
    torch_vision=False    # Use HugsVision's internal pipeline instead of torchvision datasets
)

# Returned objects:
#   - `train`: training dataset with augmentations applied.
#   - `val`: validation dataset without training-time augmentations.
#   - `id2label`: dict mapping numeric class IDs (as strings) -> human-readable labels
#   - `label2id`: inverse mapping, labels -> numeric IDs (as strings), for model config.
# Note: The string-typed IDs align with HuggingFace Transformers' configuration format.


Split Datasets...
Balance train dataset...
The less represented label in train as 100 occurrences
Size of train after balancing is 300
Training Dataset Elements:  270
+---------+---------------+-------+-------+-------+
| Dataset | affenpinscher | akita | corgi | Total |
+---------+---------------+-------+-------+-------+
|  Train  |      89       |  91   |  90   |  270  |
|  Test   |      11       |   9   |  10   |  30   |
+---------+---------------+-------+-------+-------+


In [11]:
huggingface_model = 'google/vit-base-patch16-224-in21k'

In [ ]:
# -- Model initialization -------------------------------------------------------
# Instantiate a Vision Transformer (ViT) for image classification using a
# pretrained backbone as initialization. The classification head is re-shaped
# to match the number of target classes in the current dataset. Passing
# `label2id` and `id2label` ensures that training/evaluation logs and saved
# checkpoints preserve a consistent mapping between numeric IDs and string labels.
model = ViTForImageClassification.from_pretrained(
    huggingface_model,          # e.g., "google/vit-base-patch16-224-in21k"
    num_labels=len(label2id),   # number of classes in this dataset
    label2id=label2id,          # label string -> ID (as string)
    id2label=id2label           # ID (as string) -> label string
)

# Use the unified image processor (preferred over deprecated FeatureExtractor).
# This object handles resizing, normalization, and other preprocessing steps
# consistent with the ViT backbone, thereby preventing configuration drift.
processor = AutoImageProcessor.from_pretrained(huggingface_model)

# -- Trainer configuration ------------------------------------------------------
# Configure the high-level training wrapper. The HugsVision trainer composes
# HuggingFace's `Trainer` with sensible defaults for vision tasks. Key aspects:
#   - `max_epochs`: total training epochs.
#   - `batch_size`: per-device batch size for both train and eval.
#   - `lr`: base learning rate for AdamW.
#   - `fp16`: enable mixed-precision training when a compatible GPU is available.
#   - `feature_extractor`: accepts the processor; naming remains for backward compatibility.
#   - `eval_metric="eval_loss"`: select the best model using validation loss, which is
#     always available; this avoids failures when accuracy/F1 are not computed.
trainer = VisionClassifierTrainer(
    model_name        = "a9zin",
    train             = train,
    test              = val,
    output_dir        = "./out/",
    max_epochs        = 20,
    batch_size        = 4,
    lr                = 2e-5,
    fp16              = True,
    model             = model,
    feature_extractor = processor,   # unified preprocessor for ViT
    eval_metric       = "eval_loss", # robust selection metric present by default
)


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


{'0': 'affenpinscher', '1': 'akita', '2': 'corgi'}
{'affenpinscher': '0', 'akita': '1', 'corgi': '2'}
Trainer builded!
Start Training!


Epoch,Training Loss,Validation Loss
1,No log,0.350167
2,No log,0.110412
3,No log,0.068396
4,No log,0.053194
5,No log,0.046225
6,No log,0.042183
7,No log,0.039810
8,0.145200,0.037672
9,0.145200,0.037325
10,0.145200,0.037085


Model saved at: ./out/A9ZIN/20_2025-09-28-01-03-43


In [ ]:
# -- Model evaluation -----------------------------------------------------------
# Evaluate the fine-tuned model on the held-out validation split and collect
# aligned ground-truth and predicted labels. Internally this routine runs the
# model in evaluation mode, aggregates per-batch outputs, and computes F1.
# Returns:
#   y_true : list[int] — gold labels as numeric IDs
#   y_pred : list[int] — predicted labels as numeric IDs
# These vectors can subsequently be mapped to human-readable names via `id2label`
# for reporting, or fed into `confusion_matrix` / `classification_report`.
y_true, y_pred = trainer.evaluate_f1_score()

# Example (optional, for later reporting):
# from sklearn.metrics import classification_report, confusion_matrix
# y_true_names = [id2label[str(i)] for i in y_true]
# y_pred_names = [id2label[str(i)] for i in y_pred]
# print(classification_report(y_true_names, y_pred_names))
# print(confusion_matrix(y_true_names, y_pred_names))


100%|██████████| 30/30 [00:01<00:00, 28.40it/s]

               precision    recall  f1-score   support

affenpinscher     1.0000    0.9091    0.9524        11
        akita     1.0000    1.0000    1.0000         9
        corgi     0.9091    1.0000    0.9524        10

     accuracy                         0.9667        30
    macro avg     0.9697    0.9697    0.9683        30
 weighted avg     0.9697    0.9667    0.9667        30

Logs saved at: ./out/A9ZIN/20_2025-09-28-01-03-43


In [ ]:
# -- Confusion matrix and visualization ----------------------------------------
# Compute the confusion matrix over validation predictions. By default,
# `confusion_matrix` infers the set and order of labels from `y_true`.
# This can lead to axis misalignment if we later annotate with class names
# in a different order. To ensure a fixed axis order, it is safer to pass
# an explicit `labels=` argument (see the optional block below).
cm = confusion_matrix(y_true, y_pred)

# Build a labeled DataFrame for a readable heatmap. The axis tick labels here
# are taken from `label2id.keys()`, which reflect the string class names.
# Note: dictionary insertion order is preserved in modern Python, but it may
# not match the implicit ordering used by `confusion_matrix` above. If exact
# alignment is critical, prefer the optional explicit-labels variant below.
labels = list(label2id.keys())
df_cm = pd.DataFrame(cm, index=labels, columns=labels)

# Plot a heatmap for qualitative error analysis. Annotations inside each cell
# indicate the corresponding count. A moderate figure size improves readability;
# adjust as needed based on the number of classes.
plt.figure(figsize=(10, 7))
sns.heatmap(df_cm, annot=True, annot_kws={"size": 8}, fmt="")

# Persist the figure for reports and reproducibility.
plt.savefig("./conf_matrix_1.jpg")


# --- Optional (safer axis alignment): explicit label ordering ------------------
# The following variant guarantees that matrix rows/columns align with class IDs.
# It constructs the confusion matrix with an explicit numeric label order and
# then maps those IDs to human-readable names for the DataFrame display.

# ids_in_order = list(range(len(id2label)))  # [0, 1, 2, ...]
# cm_fixed = confusion_matrix(y_true, y_pred, labels=ids_in_order)
# labels_ordered = [id2label[str(i)] for i in ids_in_order]
# df_cm_fixed = pd.DataFrame(cm_fixed, index=labels_ordered, columns=labels_ordered)
# plt.figure(figsize=(10, 7))
# sns.heatmap(df_cm_fixed, annot=True, annot_kws={"size": 8}, fmt="")
# plt.savefig("./conf_matrix_fixed.jpg")


In [ ]:
# -- Local checkpoint paths for inference --------------------------------------
# Define the base directory that contains the saved processor configuration and
# the fine-tuned model weights produced during training. The HugsVision trainer
# stores the image preprocessor ("feature_extractor") and the model in separate
# subfolders to mirror HuggingFace's expected layout.
base = "./out/A9ZIN/20_2025-09-28-01-03-43"
path_feat  = f"{base}/feature_extractor"  # directory with preprocessor_config.json
path_model = f"{base}/model"              # directory with model.safetensors + config.json

# Specify a test image for single-sample inference. The classifier expects a path
# to a file on disk; the underlying processor will handle resizing and normalization.
img = "./test/affenpinscher/affenpinscher_0.jpg"

# -- Inference wrapper construction --------------------------------------------
# Build a lightweight inference pipeline:
#   - AutoImageProcessor ensures preprocessing is consistent with the ViT backbone.
#   - ViTForImageClassification loads the fine-tuned classification head.
# The VisionClassifierInference wrapper handles preprocessing + forward pass and
# returns a human-readable class label according to the saved id2label mapping.
classifier = VisionClassifierInference(
    feature_extractor = AutoImageProcessor.from_pretrained(path_feat),
    model = ViTForImageClassification.from_pretrained(path_model),
)

# -- Single-image prediction ----------------------------------------------------
# Run a forward pass on the specified image and print the predicted class name.
# The output corresponds to the argmax over the model's logits, mapped through
# the model's `config.id2label`. Device selection (CPU/GPU) is handled internally.
label = classifier.predict(img_path=img)
print("Predicted class:", label)

# Optional sanity checks (uncomment if needed):
# import os
# assert os.path.isdir(path_feat) and os.path.isfile(os.path.join(path_feat, "preprocessor_config.json"))
# assert os.path.isdir(path_model) and os.path.isfile(os.path.join(path_model, "config.json"))
# assert os.path.isfile(img), f"Image not found: {img}"


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.50, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Model loaded!
Predicted class: affenpinscher


In [ ]:
# -- Build a dataset object directly from the ./test/ directory -----------------
# The loader expects a class-per-folder structure:
#   ./test/
#     class_a/
#       img_1.jpg, ...
#     class_b/
#       ...
#
# Return signature of `fromImageFolder` is:
#   (train_dataset, val_dataset, id2label, label2id)
# Setting `test_ratio=0` disables the internal split; the entire folder is
# returned in the *first* position. Hence `test` below holds the dataset.
#
# Important evaluation note:
#   - `augmentation=False` is generally preferred for evaluation to avoid
#     distribution shift. Data balancing should also be disabled at test time
#     to preserve the natural class frequencies. Kept here as in the current
#     configuration for continuity.
test, _, id2label, label2id = VisionDataset.fromImageFolder(
    "./test/",        # root containing one subfolder per class
    test_ratio   = 0, # no split; return the entire set in the first element
    balanced     = True,   # consider setting to False for unbiased evaluation
    augmentation = True,   # consider setting to False to avoid test-time transforms
    torch_vision = False
)

# -- Single-image prediction sanity check ---------------------------------------
# Run an inference pass on a previously defined `img` path. This confirms the
# end-to-end pipeline (preprocessing + model forward + label decoding) works.
# For full-set evaluation, iterate over `test` and aggregate predictions.
classifier.predict(img)


Split Datasets...
Balance train dataset...
The less represented label in train as 20 occurrences
Size of train after balancing is 60
Training Dataset Elements:  60
+---------+---------------+-------+-------+-------+
| Dataset | affenpinscher | akita | corgi | Total |
+---------+---------------+-------+-------+-------+
|  Train  |      20       |  20   |  20   |  60   |
|  Test   |       0       |   0   |   0   |   0   |
+---------+---------------+-------+-------+-------+


'affenpinscher'

In [ ]:
# -- Enumerate test images and run per-file inference ---------------------------
# Recursively traverse the ./test directory and collect file paths. This approach
# assumes a class-per-folder structure (e.g., ./test/<class_name>/<image>.jpg).
# Note: this pattern includes any file, not only images. For stricter filtering,
# consider restricting to known extensions, e.g., ("*.jpg", "*.jpeg", "*.png").
test_files = [f for f in glob.glob("./test/**/*", recursive=True) if os.path.isfile(f)]

# Report the total number of discovered files to aid reproducibility and debugging.
print(f"Found {len(test_files)} test images.")

# Iterate over each discovered file, infer the ground-truth class label from the
# parent directory name, and obtain the predicted label from the classifier.
# This provides a simple, human-readable log for spot-checking performance.
for fpath in test_files:
    # Derive the reference (true) label from the immediate parent directory.
    # This relies on the convention: ./test/<class_name>/<image_filename>
    true_label = os.path.basename(os.path.dirname(fpath))

    # Perform a single-image forward pass through the preprocessing pipeline
    # and the fine-tuned ViT classifier; returns a human-readable label.
    pred = classifier.predict(img_path=fpath)

    # Align the output for readability; the field width (15) can be adjusted
    # based on the longest class name in the dataset.
    print(f"{fpath} | true: {true_label:15s} | pred: {pred}")

# --- Optional robustness notes -------------------------------------------------
# • To avoid non-image files being processed, filter by extension:
#     import itertools
#     exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")
#     test_files = list(itertools.chain.from_iterable(
#         glob.glob(f"./test/**/{pat}", recursive=True) for pat in exts
#     ))
#
# • For deterministic ordering (useful for debugging/CI), sort the file list:
#     test_files = sorted(test_files)
#
# • If class folders can be nested deeper, derive `true_label` accordingly or
#   store a mapping from file paths to labels generated during dataset creation.


Found 60 test images.
./test\affenpinscher\affenpinscher_0.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_1.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_10.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_11.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_12.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_13.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_14.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_15.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_16.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_17.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_18.jpg | true: affenpinscher   | pred: affenpinscher
./test\affen